In [10]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.exc import SQLAlchemyError
from sqlalchemy.orm import sessionmaker

In [ ]:
engine = create_engine('sqlite:///../data/db/construction.db')

In [15]:
inspector = inspect(engine)

# Получить все таблицы
tables = inspector.get_table_names()
tables

['contractors', 'objects', 'progress', 'works']

In [18]:
# Для каждой таблицы вывести колонки
for table_name in tables:
    print(f"\n▶️ Таблица: {table_name}")
    columns = inspector.get_columns(table_name)
    for col in columns:
        print(f"  • {col['name']} | {col['type']} | nullable={col['nullable']}")
    
    # Внешние ключи
    fks = inspector.get_foreign_keys(table_name)
    for fk in fks:
        print(f"  ↳ FK: {fk['constrained_columns']} → {fk['referred_table']}")
    
    # Индексы
    indexes = inspector.get_indexes(table_name)
    for idx in indexes:
        print(f"  🔑 INDEX: {idx['name']} ({', '.join(idx['column_names'])})")


▶️ Таблица: contractors
  • id | INTEGER | nullable=True
  • name | TEXT | nullable=False
  • work_id | INTEGER | nullable=False
  ↳ FK: ['work_id'] → works

▶️ Таблица: objects
  • id | INTEGER | nullable=True
  • name | TEXT | nullable=False
  • city | TEXT | nullable=False
  • budget | REAL | nullable=False

▶️ Таблица: progress
  • id | INTEGER | nullable=True
  • work_id | INTEGER | nullable=False
  • plan_vol | REAL | nullable=False
  • fact_vol | REAL | nullable=False
  • date | TEXT | nullable=False
  ↳ FK: ['work_id'] → works

▶️ Таблица: works
  • id | INTEGER | nullable=True
  • object_id | INTEGER | nullable=False
  • work_type | TEXT | nullable=False
  • unit | TEXT | nullable=False
  ↳ FK: ['object_id'] → objects


In [ ]:
with engine.connect() as conn:
    query = """
    SELECT * 
    FROM works
    JOIN contractors ON works.id = contractors.work_id
    JOIN objects ON works.object_id = objects.id
    JOIN progress ON works.id = progress.work_id
    WHERE 
        objects.name = 'ЖК Панорама 23' AND 
        objects.city = 'Санкт-Петербург' AND 
        progress.plan_vol > progress.fact_vol AND 
        contractors.name = 'ООО Новый Век'
    """

    result = conn.execute(text(query))

print(result.keys())
result.fetchall()

RMKeyView(['id', 'object_id', 'work_type', 'unit', 'id', 'name', 'work_id', 'id', 'name', 'city', 'budget', 'id', 'work_id', 'plan_vol', 'fact_vol', 'date'])


[(63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 277, 63, 271.79, 124.19, '2024-01-30'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 405, 63, 281.24, 185.88, '2024-10-16'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 368, 63, 412.6, 161.62, '2024-07-08'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 204, 63, 837.28, 793.2, '2024-11-21'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 181, 63, 903.98, 18.56, '2024-04-16')]